# Healthcare Clinical Intelligence Multi-Agent System
## Week 8 Assignment: Production Deployment & UI

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Learning Objectives

- Create Gradio clinical interface
- Deploy as a FastAPI application
- Complete production readiness steps

## Difficulty: Advanced

**IMPORTANT**: Run all prior week cells first before starting this week's exercises.

---

---
# WEEK 1: Environment Setup & Clinical AI Foundations (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

## 8-Week Curriculum Overview

| Week | Module | Key Concepts |
|------|--------|-------------|
| **1** | Environment & Foundations | Setup, Langfuse, clinical data structures |
| **2** | Clinical Document Processing | Parsing DOCX/PDF, entity extraction, NER |
| **3** | Vector Stores & Embeddings | ChromaDB, semantic search, RAG foundations |
| **4** | Single Agent Design | Lab interpreter, radiology analyzer agents |
| **5** | Multi-Agent Orchestration | Role-based routing, parallel execution |
| **6** | Medical Knowledge Graphs | Ontologies, reasoning, cross-reactivity |
| **7** | Advanced Observability | Custom metrics, quality scoring, alerts |
| **8** | Production Integration | Full system, Gradio UI, deployment |

---

---

# WEEK 1: Environment Setup & Clinical AI Foundations

---

## Learning Objectives

By the end of Week 1, you will:
- Configure a Colab-compatible development environment
- Initialize Langfuse observability from the very first cell
- Understand healthcare data structures and clinical document types
- Explore the dataset with full tracing

---

## Key Concept: Why Observability Matters in Healthcare AI

Healthcare AI systems require **complete audit trails** for:

1. **Regulatory Compliance** (HIPAA, FDA)
   - Every AI recommendation must be traceable
   - Audit logs must show what data influenced decisions

2. **Clinical Safety**
   - When an AI makes a recommendation, clinicians need to understand why
   - Errors must be traceable to their root cause

3. **Quality Improvement**
   - Track accuracy over time
   - Identify patterns in failures

**Langfuse** provides this observability layer, logging every:
- LLM call with inputs/outputs
- Embedding generation
- Agent decision
- Token usage and latency

---

## 1.1 Package Installation

We install all required packages in a single cell for Colab compatibility.

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "pdfplumber", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("pdfplumber", "pdfplumber"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

## 1.2 Core Imports

We organize imports by category for clarity.

In [ ]:
# ============================================================================
# WEEK 1.2: CORE IMPORTS
# ============================================================================

# Standard library
import os
import sys
import json
import re
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple, Union
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np

print("Core imports loaded successfully")

## 1.3 Environment Detection

This notebook works in both Google Colab and local environments.

In [ ]:
# ============================================================================
# WEEK 1.3: ENVIRONMENT DETECTION, API KEY CONFIGURATION & DATA INGESTION
# ============================================================================
# API keys from Colab Secrets or .env file
# Datasets are fetched programmatically from GitHub instead of manual download.

import subprocess

IN_COLAB = 'google.colab' in sys.modules

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "healthcare"

if IN_COLAB:
    print("Running in Google Colab")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"Cloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("Dataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Running in local environment")
    from dotenv import load_dotenv
    load_dotenv()
    print("API keys loaded from .env file")

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"Cloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("Dataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

## 1.5 Langfuse Initialization

Initialize Langfuse **before any other operations** to ensure complete tracing.

In [ ]:
# ============================================================================
# WEEK 1.5: LANGFUSE INITIALIZATION
# ============================================================================

from langfuse import Langfuse
try:
    from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
except (ImportError, ModuleNotFoundError):
    try:
        from langfuse.langchain import CallbackHandler as LangfuseCallbackHandler
    except (ImportError, ModuleNotFoundError):
        # Langfuse v3+: CallbackHandler not needed, create a no-op
        class LangfuseCallbackHandler:
            def __init__(self, **kwargs): pass
from openai import OpenAI

# Initialize Langfuse client
langfuse = Langfuse(
    secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
    public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
    host=os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')
)

# Verify connection
langfuse.auth_check()
print("Langfuse connection verified!")

# Create unique session ID for this notebook run
SESSION_ID = f"healthcare-master-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
PROJECT_NAME = "healthcare-clinical-intelligence"

# Initialize OpenAI client
openai_client = OpenAI()

print(f"\nSession ID: {SESSION_ID}")
print(f"Project: {PROJECT_NAME}")
print(f"Dashboard: {os.environ.get('LANGFUSE_HOST')}/project")

## Deep Dive: Langfuse Tracing Architecture

Understanding how Langfuse organizes traces is essential for healthcare compliance:

```
Session (notebook run)
  |
  +-- Trace (single operation, e.g., "lab-interpretation")
        |
        +-- Span (sub-operation, e.g., "retrieve-context")
        |     |
        |     +-- Span (nested, e.g., "embedding-generation")
        |
        +-- Generation (LLM call with full details)
              - Model used
              - Input messages
              - Output response
              - Token usage
              - Latency
```

**For HIPAA Compliance:**
- Traces can be tagged with patient IDs (pseudonymized)
- All decisions are auditable
- Retention policies can be configured in Langfuse

---

## 1.6 Traced Utility Functions

These functions wrap OpenAI calls with Langfuse tracing. We'll use them throughout the notebook.

In [ ]:
# ============================================================================
# WEEK 1.6: TRACED EMBEDDING FUNCTION
# ============================================================================

def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    Every embedding operation is logged for observability,
    which is critical for understanding retrieval behavior.

    Args:
        text: The text to embed
        trace_name: Name for the trace (appears in Langfuse)

    Returns:
        List of floats representing the embedding vector
    """
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={"text_length": len(text), "text_preview": text[:100]}
    )

    generation = trace.generation(
        name="openai-embedding",
        model="text-embedding-3-small",
        input=text[:500]  # Log preview for debugging
    )

    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    embedding = response.data[0].embedding

    generation.end(
        output={"dimensions": len(embedding)},
        usage={"total_tokens": response.usage.total_tokens}
    )

    return embedding

print("traced_embedding() function defined")

In [ ]:
# ============================================================================
# WEEK 1.6: TRACED COMPLETION FUNCTION
# ============================================================================

def traced_completion(
    prompt: str,
    system: str = "",
    trace_name: str = "completion",
    model: str = "gpt-4o-mini",
    temperature: float = 0,
    metadata: Dict = None
) -> str:
    """
    Generate LLM completion with full Langfuse tracing.

    Every LLM call is logged with:
    - Input messages
    - Output response
    - Token usage (for cost tracking)
    - Model and parameters

    Args:
        prompt: User prompt
        system: System message (optional)
        trace_name: Name for the trace
        model: OpenAI model to use
        temperature: Sampling temperature
        metadata: Additional metadata to log

    Returns:
        The LLM's response text
    """
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={"model": model, "temperature": temperature, **(metadata or {})}
    )

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    generation = trace.generation(
        name="openai-completion",
        model=model,
        input=messages
    )

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )

    result = response.choices[0].message.content

    generation.end(
        output=result,
        usage={
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    )

    return result

print("traced_completion() function defined")

In [ ]:
# ============================================================================
# WEEK 1.6: LANGCHAIN CALLBACK HANDLER
# ============================================================================

def get_langfuse_handler(
    trace_name: str,
    tags: List[str] = None,
    metadata: Dict = None
) -> LangfuseCallbackHandler:
    """
    Get a LangChain callback handler for automatic tracing.

    Use this with LangChain LCEL chains for automatic
    tracing of all chain operations.

    Args:
        trace_name: Name for the trace
        tags: List of tags for filtering in Langfuse
        metadata: Additional metadata

    Returns:
        LangfuseCallbackHandler instance
    """
    return LangfuseCallbackHandler(
        secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
        public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
        host=os.environ.get('LANGFUSE_HOST'),
        session_id=SESSION_ID,
        trace_name=trace_name,
        tags=tags or [],
        metadata=metadata or {}
    )

print("get_langfuse_handler() function defined")
print("\nTraced utility functions ready:")
print("  - traced_embedding(): Generate embeddings with tracing")
print("  - traced_completion(): LLM completions with tracing")
print("  - get_langfuse_handler(): LangChain callback handler")

## Key Concept: Understanding Clinical Laboratory Values

Before we process clinical data, let's understand the key lab tests we'll encounter:

### Complete Blood Count (CBC)

| Test | Normal Range | Low Indicates | High Indicates |
|------|-------------|---------------|----------------|
| **Hemoglobin** | M: 13.5-17.5 g/dL, F: 12-16 g/dL | Anemia, bleeding | Dehydration, polycythemia |
| **WBC** | 4,500-11,000 /uL | Infection risk, bone marrow issues | Infection, leukemia |
| **Platelets** | 150,000-400,000 /uL | Bleeding risk | Clotting risk |
| **RBC** | M: 4.5-5.5 M/uL, F: 4.0-5.0 M/uL | Anemia | Dehydration, polycythemia |

### Liver Function Tests (LFT)

| Test | Normal Range | Elevated Indicates |
|------|-------------|--------------------|
| **AST (SGOT)** | 10-40 U/L | Liver damage, heart attack, muscle injury |
| **ALT (SGPT)** | 7-56 U/L | Liver-specific damage |
| **Bilirubin** | 0.1-1.2 mg/dL | Liver disease, bile duct obstruction |
| **Albumin** | 3.5-5.0 g/dL | Liver disease, malnutrition |

### Kidney Function Tests (KFT)

| Test | Normal Range | Abnormal Indicates |
|------|-------------|--------------------|
| **Creatinine** | 0.7-1.3 mg/dL | Kidney function impairment |
| **BUN** | 7-20 mg/dL | Kidney disease, dehydration |
| **eGFR** | >90 mL/min | Lower = worse kidney function |

### Thyroid Panel

| Test | Normal Range | Low | High |
|------|-------------|-----|------|
| **TSH** | 0.4-4.0 mIU/L | Hyperthyroidism | Hypothyroidism |
| **Free T4** | 0.8-1.8 ng/dL | Hypothyroidism | Hyperthyroidism |

---

## 1.7 Data Discovery with Tracing

Let's explore the clinical dataset with full observability.

In [ ]:
# ============================================================================
# WEEK 1.7: DATA DISCOVERY FUNCTION
# ============================================================================

def discover_clinical_data(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Discover all clinical documents organized by category.

    Healthcare data is typically organized by:
    - Role (patient, clinician, radiologist)
    - Type (labs, imaging, allergies)
    - Specialty (food allergy, medication allergy, etc.)

    Returns:
        Dictionary mapping category names to file paths
    """
    categories = {
        'patient': [],
        'clinician': [],
        'radiologist': [],
        'training': [],
        'allergy_food': [],
        'allergy_medication': [],
        'allergy_environment': [],
        'allergy_skin': []
    }

    if not data_dir.exists():
        print(f"Warning: Data directory not found at {data_dir}")
        return categories

    for item in data_dir.rglob('*'):
        if item.is_file() and item.suffix.lower() in ['.docx', '.pdf', '.xlsx']:
            path_str = str(item).lower()

            # Categorize by path content
            if 'food allergy' in path_str:
                categories['allergy_food'].append(item)
            elif 'medication allergy' in path_str:
                categories['allergy_medication'].append(item)
            elif 'environment allergy' in path_str:
                categories['allergy_environment'].append(item)
            elif 'skin allergy' in path_str:
                categories['allergy_skin'].append(item)
            elif 'clinician' in path_str or 'doctor' in path_str:
                categories['clinician'].append(item)
            elif 'radiologist' in path_str:
                categories['radiologist'].append(item)
            elif 'patient' in path_str:
                categories['patient'].append(item)
            elif 'training' in path_str:
                categories['training'].append(item)

    return categories

print("discover_clinical_data() function defined")

In [ ]:
# ============================================================================
# WEEK 1.7: EXECUTE DATA DISCOVERY WITH TRACING
# ============================================================================

# Trace the data discovery operation
discovery_trace = langfuse.trace(
    name="data-discovery",
    session_id=SESSION_ID,
    metadata={"data_dir": str(DATA_DIR)},
    tags=["week1", "setup"]
)

clinical_files = discover_clinical_data(DATA_DIR)

# Log discovery results
discovery_summary = {cat: len(files) for cat, files in clinical_files.items()}
discovery_trace.update(output=discovery_summary)

print("\n" + "="*60)
print("HEALTHCARE DATASET DISCOVERY")
print("="*60)

total_files = 0
for category, files in clinical_files.items():
    if files:
        print(f"\n{category.upper().replace('_', ' ')}: {len(files)} files")
        for f in files[:3]:
            print(f"  - {f.name}")
        if len(files) > 3:
            print(f"  ... and {len(files) - 3} more")
        total_files += len(files)

print(f"\nTotal clinical documents discovered: {total_files}")

## 1.8 Exploratory Data Analysis

Visualize the dataset composition to understand what we're working with.

In [ ]:
# ============================================================================
# WEEK 1.8: BUILD EDA DATAFRAME
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Trace EDA
eda_trace = langfuse.trace(
    name="exploratory-data-analysis",
    session_id=SESSION_ID,
    tags=["week1", "eda", "visualization"]
)

# Build EDA dataframe
eda_data = []
for category, files in clinical_files.items():
    for f in files:
        try:
            size_kb = f.stat().st_size / 1024
        except:
            size_kb = 0
        eda_data.append({
            'category': category.replace('_', ' ').title(),
            'filename': f.name,
            'extension': f.suffix.lower(),
            'size_kb': size_kb
        })

eda_df = pd.DataFrame(eda_data)

print("DATASET SUMMARY")
print("="*60)
print(f"Total documents: {len(eda_df)}")
print(f"\nDocuments by category:")
if len(eda_df) > 0:
    print(eda_df['category'].value_counts().to_string())
    print(f"\nFile types:")
    print(eda_df['extension'].value_counts().to_string())

# Log EDA results
eda_trace.update(
    output={
        "total_documents": len(eda_df),
        "categories": eda_df['category'].value_counts().to_dict() if len(eda_df) > 0 else {},
        "file_types": eda_df['extension'].value_counts().to_dict() if len(eda_df) > 0 else {}
    }
)

In [ ]:
# ============================================================================
# WEEK 1.8: EDA VISUALIZATIONS
# ============================================================================

if len(eda_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 1. Documents by category
    category_counts = eda_df['category'].value_counts()
    axes[0].barh(category_counts.index, category_counts.values, color='steelblue')
    axes[0].set_xlabel('Number of Documents')
    axes[0].set_title('Documents by Category')
    axes[0].invert_yaxis()

    # 2. File types distribution
    ext_counts = eda_df['extension'].value_counts()
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
    axes[1].pie(ext_counts.values, labels=ext_counts.index, autopct='%1.1f%%',
                colors=colors[:len(ext_counts)])
    axes[1].set_title('File Types Distribution')

    # 3. File sizes by category
    if eda_df['size_kb'].sum() > 0:
        size_by_cat = eda_df.groupby('category')['size_kb'].sum().sort_values(ascending=True)
        axes[2].barh(size_by_cat.index, size_by_cat.values, color='coral')
        axes[2].set_xlabel('Total Size (KB)')
        axes[2].set_title('Total File Size by Category')
    else:
        axes[2].text(0.5, 0.5, 'Size data not available', ha='center', va='center')
        axes[2].set_title('File Sizes')

    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization")
    print("Ensure the data directory path is correct.")

## Checkpoint: Week 1 Verification

Before proceeding to Week 2, verify that all Week 1 components are working.

In [ ]:
# ============================================================================
# WEEK 1 CHECKPOINT
# ============================================================================

print("WEEK 1 CHECKPOINT")
print("="*60)

checks = {
    "Langfuse initialized": langfuse is not None,
    "OpenAI client ready": openai_client is not None,
    "Session ID created": SESSION_ID is not None,
    "Data discovery complete": clinical_files is not None,
    "EDA dataframe created": eda_df is not None
}

all_passed = True
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")
    if not passed:
        all_passed = False

if all_passed:
    print("\nAll Week 1 checks passed! Ready for Week 2.")
else:
    print("\nSome checks failed. Review the cells above.")

## Week 1 Summary

### What We Accomplished

1. **Environment Setup**
   - Installed all required packages
   - Configured API keys securely
   - Set up Colab/local compatibility

2. **Langfuse Observability**
   - Initialized Langfuse client
   - Created session ID for tracing
   - Built traced utility functions

3. **Clinical Data Exploration**
   - Discovered dataset structure
   - Visualized document distribution
   - Understood clinical categories

### Key Concepts Learned

- **Observability**: Every operation should be traced for healthcare compliance
- **Lab Values**: Understanding normal ranges for CBC, LFT, KFT, thyroid
- **Data Organization**: Healthcare data organized by role and specialty

### Next Week Preview

In Week 2, we'll learn to **parse clinical documents** and **extract medical entities** from unstructured text.

---

---
# WEEK 2: Clinical Document Processing & Entity Extraction (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 2: Clinical Document Processing & Entity Extraction

---

## Learning Objectives

By the end of Week 2, you will:
- Parse heterogeneous clinical documents (DOCX, PDF, XLSX)
- Extract structured information from unstructured clinical text
- Implement Named Entity Recognition (NER) for medical terms
- Build a document processing pipeline with full tracing

---

## Key Concept: Clinical Document Types

Healthcare systems generate diverse document types, each with unique structures:

### 1. Laboratory Reports
- **Format**: Usually tabular (test name, value, unit, reference range)
- **Challenge**: Values need context (patient age, sex, medications)
- **Example**: CBC showing Hemoglobin 10.2 g/dL

### 2. Radiology Reports
- **Format**: Narrative text with sections (Indication, Technique, Findings, Impression)
- **Challenge**: Medical terminology, abbreviations
- **Example**: "Chest X-ray shows bilateral infiltrates consistent with pneumonia"

### 3. Clinical Notes
- **Format**: Free text, often with templates (SOAP notes)
- **Challenge**: Abbreviations, inconsistent formatting
- **Example**: "Pt c/o SOB x 3d, denies CP"

### 4. Allergy Records
- **Format**: Structured lists or narrative
- **Challenge**: Cross-reactivity relationships
- **Example**: "Penicillin allergy - anaphylaxis 2019"

---

## Deep Dive: HIPAA Considerations in Document Processing

When processing clinical documents, HIPAA compliance is critical:

### Protected Health Information (PHI)

The 18 HIPAA identifiers that must be protected:

1. Names
2. Geographic data (smaller than state)
3. Dates (except year) related to an individual
4. Phone numbers
5. Fax numbers
6. Email addresses
7. Social Security numbers
8. Medical record numbers
9. Health plan beneficiary numbers
10. Account numbers
11. Certificate/license numbers
12. Vehicle identifiers
13. Device identifiers
14. URLs
15. IP addresses
16. Biometric identifiers
17. Full-face photographs
18. Any unique identifying number

### Best Practices for AI Systems

1. **De-identification**: Remove PHI before processing when possible
2. **Audit Trails**: Log all access to PHI (Langfuse helps here!)
3. **Minimum Necessary**: Only access data needed for the task
4. **Encryption**: Encrypt data at rest and in transit

---

In [ ]:
# ============================================================================
# WEEK 2.1: DOCUMENT LOADING IMPORTS
# ============================================================================

from docx import Document as DocxDocument
import pdfplumber

print("Document processing libraries loaded")

In [ ]:
# ============================================================================
# WEEK 2.2: CLINICAL DOCUMENT PROCESSOR CLASS
# ============================================================================

class ClinicalDocumentProcessor:
    """
    Production-grade document processor for healthcare documents.

    Handles multiple formats:
    - DOCX: Word documents (common for clinical notes)
    - PDF: Lab reports, radiology reports
    - XLSX: Structured lab data

    All operations are traced to Langfuse for audit compliance.
    """

    def __init__(self):
        self.supported_formats = ['.docx', '.pdf', '.xlsx']
        self.processed_docs = []

    def load_docx(self, file_path: Path) -> Dict[str, Any]:
        """
        Load and parse a DOCX file.

        Extracts:
        - Paragraphs (narrative text)
        - Tables (structured data like lab values)
        """
        try:
            doc = DocxDocument(file_path)
            paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

            # Extract tables (common in lab reports)
            tables = []
            for table in doc.tables:
                table_data = []
                for row in table.rows:
                    row_data = [cell.text.strip() for cell in row.cells]
                    if any(row_data):
                        table_data.append(row_data)
                if table_data:
                    tables.append(table_data)

            return {
                'filename': file_path.name,
                'format': 'docx',
                'paragraphs': paragraphs,
                'tables': tables,
                'full_text': '\n'.join(paragraphs),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading {file_path.name}: {e}")
            return None

    def load_pdf(self, file_path: Path) -> Dict[str, Any]:
        """
        Load and parse a PDF file.

        Uses pdfplumber for better table extraction.
        """
        try:
            text_content = []
            tables = []

            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        text_content.append(text)
                    page_tables = page.extract_tables()
                    if page_tables:
                        tables.extend(page_tables)

            return {
                'filename': file_path.name,
                'format': 'pdf',
                'paragraphs': text_content,
                'tables': tables,
                'full_text': '\n'.join(text_content),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading PDF {file_path.name}: {e}")
            return None

    def _tables_to_text(self, tables: List) -> str:
        """Convert tables to readable text format."""
        text_parts = []
        for i, table in enumerate(tables):
            text_parts.append(f"\nTable {i+1}:")
            for row in table:
                if isinstance(row, list):
                    text_parts.append(' | '.join(str(cell) for cell in row if cell))
        return '\n'.join(text_parts)

    def process_file(self, file_path: Path) -> Dict[str, Any]:
        """Process a single file based on its format."""
        suffix = file_path.suffix.lower()

        if suffix == '.docx':
            return self.load_docx(file_path)
        elif suffix == '.pdf':
            return self.load_pdf(file_path)
        else:
            print(f"Unsupported format: {suffix}")
            return None

print("ClinicalDocumentProcessor class defined")

In [ ]:
# ============================================================================
# WEEK 2.3: TRACED DOCUMENT PROCESSING
# ============================================================================

def process_category_with_tracing(
    processor: ClinicalDocumentProcessor,
    files: List[Path],
    category: str
) -> List[Dict]:
    """
    Process all files in a category with Langfuse tracing.

    Each document processing is logged as a span within the trace,
    providing a complete audit trail of document ingestion.
    """
    trace = langfuse.trace(
        name=f"process-{category}",
        session_id=SESSION_ID,
        metadata={"category": category, "file_count": len(files)},
        tags=["week2", "document-processing", category]
    )

    processed = []
    for file_path in files:
        span = trace.span(name=f"process-{file_path.name}")
        doc = processor.process_file(file_path)
        if doc:
            doc['category'] = category
            doc['doc_id'] = f"{category.upper()}-{len(processed)+1:03d}"
            processed.append(doc)
            span.end(output={"success": True, "doc_id": doc['doc_id']})
        else:
            span.end(output={"success": False})

    trace.update(output={"processed_count": len(processed)})
    return processed

print("process_category_with_tracing() function defined")

In [ ]:
# ============================================================================
# WEEK 2.4: PROCESS ALL CLINICAL DOCUMENTS
# ============================================================================

# Initialize processor
doc_processor = ClinicalDocumentProcessor()

# Process all categories with tracing
all_clinical_docs = {}

print("Processing clinical documents...")
print("Each document is traced to Langfuse\n")

for category, files in clinical_files.items():
    if files:
        all_clinical_docs[category] = process_category_with_tracing(
            doc_processor, files, category
        )
        print(f"Processed {len(all_clinical_docs[category])} {category} documents")

total_processed = sum(len(docs) for docs in all_clinical_docs.values())
print(f"\nTotal documents processed: {total_processed}")

## Key Concept: Medical Named Entity Recognition (NER)

Medical NER identifies and classifies clinical entities in text:

### Entity Types in Clinical Text

| Entity Type | Examples | Importance |
|-------------|----------|------------|
| **Lab Tests** | CBC, TSH, Creatinine | Core clinical data |
| **Lab Values** | 10.5 g/dL, 150,000 /uL | Quantitative results |
| **Medications** | Metformin, Lisinopril | Drug interactions |
| **Diagnoses** | Diabetes, Hypertension | Clinical context |
| **Symptoms** | Fatigue, chest pain | Patient presentation |
| **Allergens** | Penicillin, peanuts | Safety critical |

### Approaches to Medical NER

1. **Rule-Based**: Regex patterns for structured data (lab values)
2. **Dictionary-Based**: Match against medical vocabularies (SNOMED, ICD)
3. **ML-Based**: Fine-tuned models (BioBERT, ClinicalBERT)
4. **LLM-Based**: GPT for complex extraction tasks

We'll use a **hybrid approach**: rules for structured data, LLMs for complex extraction.

---

In [ ]:
# ============================================================================
# WEEK 2.5: CLINICAL ENTITY EXTRACTOR CLASS
# ============================================================================

class ClinicalEntityExtractor:
    """
    Extract clinical entities from medical documents.

    Uses regex patterns for structured data extraction
    and prepares data for LLM-based extraction.

    All extractions are traced to Langfuse for audit compliance.
    """

    def __init__(self):
        # Lab test patterns with units
        self.lab_patterns = {
            'hemoglobin': r'(?:Hemoglobin|Hgb|HGB)[:\s]+(\d+\.?\d*)\s*(g/dL|g/dl)?',
            'wbc': r'(?:WBC|White Blood Cell)[:\s]+(\d+[,\d]*)\s*(/uL|/\u03bcL)?',
            'rbc': r'(?:RBC|Red Blood Cell)[:\s]+(\d+\.?\d*)\s*(M/uL)?',
            'platelet': r'(?:Platelet|PLT)[:\s]+(\d+[,\d]*)',
            'glucose': r'(?:Glucose|Blood Sugar)[:\s]+(\d+\.?\d*)\s*(mg/dL)?',
            'creatinine': r'(?:Creatinine)[:\s]+(\d+\.?\d*)\s*(mg/dL)?',
            'bun': r'(?:BUN|Blood Urea Nitrogen)[:\s]+(\d+\.?\d*)\s*(mg/dL)?',
            'bilirubin': r'(?:Bilirubin)[:\s]+(\d+\.?\d*)\s*(mg/dL)?',
            'ast': r'(?:AST|SGOT)[:\s]+(\d+\.?\d*)\s*(U/L)?',
            'alt': r'(?:ALT|SGPT)[:\s]+(\d+\.?\d*)\s*(U/L)?',
            'tsh': r'(?:TSH)[:\s]+(\d+\.?\d*)\s*(mIU/L)?',
            't4': r'(?:Free T4|T4)[:\s]+(\d+\.?\d*)\s*(ng/dL)?'
        }

        # Status indicators
        self.status_patterns = {
            'normal': r'\b(NORMAL|Normal|WNL|Within Normal Limits)\b',
            'high': r'\b(HIGH|High|Elevated|Increased|Above|H)\b',
            'low': r'\b(LOW|Low|Decreased|Below|L)\b',
            'critical': r'\b(CRITICAL|Critical|STAT|Urgent|PANIC)\b'
        }

        # Common allergens
        self.allergen_list = [
            # Food
            'peanut', 'wheat', 'milk', 'egg', 'soy', 'fish', 'shellfish',
            'gluten', 'tree nut', 'sesame',
            # Medication
            'penicillin', 'sulfa', 'aspirin', 'ibuprofen', 'nsaid',
            'codeine', 'morphine', 'latex',
            # Environmental
            'dust', 'pollen', 'mold', 'pet dander', 'bee', 'wasp'
        ]

    def extract_lab_values(self, text: str) -> List[Dict]:
        """Extract lab test values from text using regex patterns."""
        results = []
        for test_name, pattern in self.lab_patterns.items():
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                value = match[0] if isinstance(match, tuple) else match
                unit = match[1] if isinstance(match, tuple) and len(match) > 1 else ''
                results.append({
                    'test': test_name,
                    'value': value.replace(',', ''),
                    'unit': unit or 'N/A'
                })
        return results

    def extract_status(self, text: str) -> List[str]:
        """Extract status indicators from text."""
        statuses = []
        for status, pattern in self.status_patterns.items():
            if re.search(pattern, text):
                statuses.append(status)
        return statuses

    def extract_allergies(self, text: str) -> List[str]:
        """Extract potential allergens mentioned in text."""
        text_lower = text.lower()
        return [a for a in self.allergen_list if a in text_lower]

print("ClinicalEntityExtractor class defined")

In [ ]:
# ============================================================================
# WEEK 2.6: TRACED ENTITY EXTRACTION
# ============================================================================

def extract_entities_with_tracing(
    extractor: ClinicalEntityExtractor,
    doc: Dict
) -> Dict:
    """
    Extract all clinical entities with Langfuse tracing.

    This creates an audit trail of what was extracted
    from each clinical document.
    """
    trace = langfuse.trace(
        name=f"entity-extraction-{doc.get('doc_id', 'unknown')}",
        session_id=SESSION_ID,
        metadata={"doc_id": doc.get('doc_id'), "category": doc.get('category')},
        tags=["week2", "entity-extraction"]
    )

    # Combine all text sources
    text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')

    result = {
        'doc_id': doc.get('doc_id'),
        'category': doc.get('category'),
        'lab_values': extractor.extract_lab_values(text),
        'statuses': extractor.extract_status(text),
        'allergies': extractor.extract_allergies(text)
    }

    trace.update(output=result)
    return result

print("extract_entities_with_tracing() function defined")

In [ ]:
# ============================================================================
# WEEK 2.7: ENTITY EXTRACTION EXAMPLES - Lab Values
# ============================================================================

# Initialize extractor
entity_extractor = ClinicalEntityExtractor()

print("ENTITY EXTRACTION EXAMPLES - Laboratory Values")
print("="*60)

# Example lab reports
lab_examples = [
    """CBC Report:
    Hemoglobin: 10.2 g/dL (LOW)
    WBC: 12,500 /uL (HIGH)
    Platelet: 180,000 /uL (NORMAL)""",

    """Liver Function Test:
    AST: 85 U/L (ELEVATED)
    ALT: 92 U/L (HIGH)
    Bilirubin: 1.8 mg/dL""",

    """Kidney Function:
    Creatinine: 2.1 mg/dL (HIGH)
    BUN: 28 mg/dL (ELEVATED)""",

    """Thyroid Panel:
    TSH: 8.5 mIU/L (HIGH)
    Free T4: 0.6 ng/dL (LOW)"""
]

for i, example in enumerate(lab_examples, 1):
    print(f"\n[Example {i}]")
    print("-" * 40)

    labs = entity_extractor.extract_lab_values(example)
    statuses = entity_extractor.extract_status(example)

    print(f"Extracted lab values:")
    for lab in labs:
        print(f"  - {lab['test']}: {lab['value']} {lab['unit']}")
    print(f"Status indicators: {statuses}")

In [ ]:
# ============================================================================
# WEEK 2.8: ENTITY EXTRACTION EXAMPLES - Allergies
# ============================================================================

print("\nENTITY EXTRACTION EXAMPLES - Allergies")
print("="*60)

allergy_examples = [
    "Patient has severe peanut allergy with history of anaphylaxis.",
    "Allergies: Penicillin (rash), Sulfa drugs (hives)",
    "Environmental allergies include dust mites and pollen.",
    "Patient reports ibuprofen and aspirin cause GI upset.",
    "Food allergies: wheat, milk, egg. Carries EpiPen."
]

for i, example in enumerate(allergy_examples, 1):
    print(f"\n[Example {i}]")
    print(f"Text: {example}")
    allergies = entity_extractor.extract_allergies(example)
    print(f"Extracted allergens: {allergies}")

In [ ]:
# ============================================================================
# WEEK 2.9: PROCESS REAL CLINICAL DOCUMENTS
# ============================================================================

print("\nEXTRACTING ENTITIES FROM REAL CLINICAL DOCUMENTS")
print("="*60)
print("Each extraction is traced to Langfuse\n")

extraction_results = []
categories_to_process = ['clinician', 'allergy_food', 'allergy_medication']

for category in categories_to_process:
    if category in all_clinical_docs:
        print(f"\n--- {category.upper().replace('_', ' ')} ---")
        for doc in all_clinical_docs[category][:3]:
            entities = extract_entities_with_tracing(entity_extractor, doc)
            extraction_results.append(entities)

            print(f"\nDocument: {doc['filename']}")
            print(f"  Lab Values: {len(entities['lab_values'])} found")
            for lab in entities['lab_values'][:3]:
                print(f"    - {lab['test']}: {lab['value']} {lab['unit']}")
            print(f"  Statuses: {entities['statuses']}")
            print(f"  Allergies: {entities['allergies']}")

print(f"\nTotal extraction operations: {len(extraction_results)}")

In [ ]:
# ============================================================================
# WEEK 2 CHECKPOINT
# ============================================================================

print("WEEK 2 CHECKPOINT")
print("="*60)

checks = {
    "Document processor initialized": doc_processor is not None,
    "Documents processed": len(all_clinical_docs) > 0,
    "Entity extractor initialized": entity_extractor is not None,
    "Lab patterns defined": len(entity_extractor.lab_patterns) >= 10,
    "Allergen list defined": len(entity_extractor.allergen_list) >= 15
}

all_passed = True
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")
    if not passed:
        all_passed = False

if all_passed:
    print("\nAll Week 2 checks passed! Ready for Week 3.")
else:
    print("\nSome checks failed. Review the cells above.")

## Week 2 Summary

### What We Accomplished

1. **Document Processing Pipeline**
   - Built parser for DOCX and PDF files
   - Extracted tables and narrative text
   - Created document metadata structure

2. **Entity Extraction**
   - Implemented regex patterns for lab values
   - Built allergen detection system
   - Created status indicator extraction

3. **Tracing & Compliance**
   - Every document operation traced
   - Entity extractions logged for audit

### Key Concepts Learned

- **HIPAA**: 18 identifiers that constitute PHI
- **Clinical Documents**: Lab reports, radiology, clinical notes
- **Medical NER**: Extracting structured data from unstructured text

### Next Week Preview

In Week 3, we'll build a **vector store** for semantic search and implement **RAG** (Retrieval Augmented Generation).

---

---
# WEEK 3: Vector Stores & Semantic Search (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 3: Vector Stores & Semantic Search

---

## Learning Objectives

By the end of Week 3, you will:
- Understand how embeddings represent text semantically
- Set up ChromaDB for vector storage with full tracing
- Index clinical documents for semantic retrieval
- Implement RAG (Retrieval Augmented Generation) foundations

---

## Key Concept: Understanding Embeddings in Healthcare

Embeddings convert text into dense numerical vectors that capture semantic meaning.

### Why Embeddings Matter for Clinical AI

Traditional keyword search fails in healthcare because:
- "Heart attack" = "Myocardial infarction" = "MI" (same concept, different terms)
- "SOB" could mean "shortness of breath" or something else entirely
- Clinical context matters: "negative" can be good ("negative for cancer")

### How Embeddings Work

```
Input Text                   Embedding Vector (1536 dimensions)
"chest pain"       ->       [0.23, -0.45, 0.12, ..., 0.08]
"cardiac discomfort" ->     [0.21, -0.43, 0.14, ..., 0.07]  <- Similar!
"leg pain"         ->       [0.05, -0.22, 0.67, ..., 0.34]  <- Different
```

### Similarity Metrics

| Metric | Description | Use Case |
|--------|-------------|----------|
| **Cosine Similarity** | Angle between vectors | Most common, scale-invariant |
| **Euclidean Distance** | Straight-line distance | When magnitude matters |
| **Dot Product** | Vector multiplication | Fast computation |

We use **cosine similarity** for clinical text (typical range: 0.7-0.95 for related content).

---

In [ ]:
# ============================================================================
# WEEK 3.1: CHROMADB IMPORTS
# ============================================================================

import chromadb

print("ChromaDB imported successfully")
print(f"ChromaDB version: {chromadb.__version__}")

In [ ]:
# ============================================================================
# WEEK 3.2: TRACED CLINICAL VECTOR STORE CLASS
# ============================================================================

class TracedClinicalVectorStore:
    """
    ChromaDB-based vector store with full Langfuse tracing.

    Every embedding and search operation is logged, providing:
    - Complete audit trail for HIPAA compliance
    - Performance monitoring (latency, token usage)
    - Debugging information for retrieval quality
    """

    def __init__(self, persist_directory: str = "./chroma_healthcare_db"):
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collections = {}
        print(f"ChromaDB initialized at: {persist_directory}")

    def create_collection(self, name: str) -> chromadb.Collection:
        """
        Create or get a collection.

        Collections organize documents by type:
        - healthcare_clinician: Lab reports
        - healthcare_radiologist: Imaging reports
        - healthcare_allergy: Allergy records
        """
        collection = self.client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )
        self.collections[name] = collection
        return collection

    def chunk_document(self, text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
        """
        Split document into overlapping chunks for better retrieval.

        Why chunking matters:
        - LLMs have context limits
        - Smaller chunks = more precise retrieval
        - Overlap ensures no context is lost at boundaries
        """
        words = text.split()
        chunks = []
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)
        return chunks if chunks else [text]

print("TracedClinicalVectorStore class defined (Part 1)")

In [ ]:
# ============================================================================
# WEEK 3.3: VECTOR STORE - INDEXING METHOD
# ============================================================================

def add_documents_with_tracing(
    vector_store: TracedClinicalVectorStore,
    collection_name: str,
    documents: List[Dict]
):
    """
    Add documents to vector store with full Langfuse tracing.

    Each document is:
    1. Split into chunks
    2. Embedded (traced)
    3. Stored with metadata
    """
    trace = langfuse.trace(
        name=f"index-{collection_name}",
        session_id=SESSION_ID,
        metadata={"collection": collection_name, "doc_count": len(documents)},
        tags=["week3", "indexing", collection_name]
    )

    collection = vector_store.create_collection(collection_name)

    all_ids = []
    all_texts = []
    all_embeddings = []
    all_metadata = []

    for doc in documents:
        text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')
        chunks = vector_store.chunk_document(text)

        for i, chunk in enumerate(chunks):
            chunk_id = f"{doc['doc_id']}_chunk_{i}"

            # Generate traced embedding
            embedding = traced_embedding(chunk, f"embed-{chunk_id}")

            all_ids.append(chunk_id)
            all_texts.append(chunk)
            all_embeddings.append(embedding)
            all_metadata.append({
                'doc_id': doc['doc_id'],
                'filename': doc['filename'],
                'category': doc['category'],
                'chunk_index': i
            })

    if all_ids:
        collection.add(
            ids=all_ids,
            embeddings=all_embeddings,
            documents=all_texts,
            metadatas=all_metadata
        )
        print(f"Added {len(all_ids)} chunks to '{collection_name}'")

    trace.update(output={"chunks_added": len(all_ids)})

print("add_documents_with_tracing() function defined")

In [ ]:
# ============================================================================
# WEEK 3.4: VECTOR STORE - SEARCH METHOD
# ============================================================================

def search_with_tracing(
    vector_store: TracedClinicalVectorStore,
    collection_name: str,
    query: str,
    n_results: int = 5
) -> Dict:
    """
    Semantic search with Langfuse tracing.

    The query is embedded and compared against stored vectors
    using cosine similarity. Results are ranked by relevance.
    """
    trace = langfuse.trace(
        name=f"search-{collection_name}",
        session_id=SESSION_ID,
        input={"query": query, "n_results": n_results},
        tags=["week3", "search", collection_name]
    )

    if collection_name not in vector_store.collections:
        vector_store.create_collection(collection_name)

    collection = vector_store.collections[collection_name]

    # Generate traced query embedding
    query_embedding = traced_embedding(query, f"search-query-{collection_name}")

    # Perform search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )

    trace.update(output={
        "results_count": len(results['documents'][0]) if results['documents'] else 0
    })

    return results

print("search_with_tracing() function defined")

In [ ]:
# ============================================================================
# WEEK 3.5: INITIALIZE VECTOR STORE
# ============================================================================

# Initialize the traced vector store
vector_store = TracedClinicalVectorStore()

print("\nVector store initialized and ready for indexing")

In [ ]:
# ============================================================================
# WEEK 3.6: INDEX CLINICAL DOCUMENTS
# ============================================================================

print("INDEXING CLINICAL DOCUMENTS INTO CHROMADB")
print("="*60)
print("Each embedding operation is traced to Langfuse\n")

# Index each category separately
for category, docs in all_clinical_docs.items():
    if docs:
        add_documents_with_tracing(vector_store, f"healthcare_{category}", docs)

# Also create a unified collection
all_docs = []
for docs in all_clinical_docs.values():
    all_docs.extend(docs)

if all_docs:
    add_documents_with_tracing(vector_store, "healthcare_all", all_docs)

print("\nIndexing complete!")

## Key Concept: RAG (Retrieval Augmented Generation)

RAG combines retrieval with LLM generation for more accurate, grounded responses.

### The RAG Pipeline

```
User Query
    |
    v
[Embed Query] -> Vector Search -> [Retrieved Documents]
    |                                     |
    v                                     v
[LLM] <---- Context Augmentation <--------+
    |
    v
Grounded Response
```

### Why RAG for Healthcare?

1. **Accuracy**: Responses are grounded in actual clinical documents
2. **Traceability**: Can cite sources for clinical decisions
3. **Up-to-date**: Uses current clinical guidelines, not training data
4. **Specialization**: Works with institution-specific protocols

### RAG Quality Metrics

| Metric | Description | Target |
|--------|-------------|--------|
| **Precision** | Relevant docs in top-k | >80% |
| **Recall** | All relevant docs retrieved | >70% |
| **MRR** | Mean Reciprocal Rank | >0.7 |

---

In [ ]:
# ============================================================================
# WEEK 3.7: SEMANTIC SEARCH EXAMPLES - Clinical Queries
# ============================================================================

print("SEMANTIC SEARCH DEMONSTRATION")
print("="*60)
print("Each query is traced to Langfuse\n")

clinical_queries = [
    "What are the normal hemoglobin levels?",
    "Patient with peanut allergy symptoms",
    "Chest X-ray findings for pneumonia",
    "Liver function test interpretation"
]

for i, query in enumerate(clinical_queries, 1):
    print(f"\n[Query {i}] '{query}'")
    print("-" * 50)

    results = search_with_tracing(vector_store, "healthcare_all", query, n_results=2)

    if results['documents'] and results['documents'][0]:
        for j, (doc, meta, dist) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )):
            similarity = 1 - dist  # Convert distance to similarity
            print(f"  Result {j+1} (Similarity: {similarity:.3f})")
            print(f"  Source: {meta['filename']} ({meta['category']})")
            print(f"  Preview: {doc[:100]}...")
    else:
        print("  No results found")

In [ ]:
# ============================================================================
# WEEK 3.8: MORE SEARCH EXAMPLES - Specialty Queries
# ============================================================================

print("\nSPECIALTY-SPECIFIC SEARCH EXAMPLES")
print("="*60)

specialty_queries = [
    ("Medication allergy to ibuprofen", "healthcare_allergy_medication"),
    ("Thyroid hormone levels TSH", "healthcare_clinician"),
    ("MRI findings brain", "healthcare_radiologist"),
    ("Kidney function creatinine", "healthcare_clinician")
]

for query, collection in specialty_queries:
    print(f"\nQuery: '{query}'")
    print(f"Collection: {collection}")

    try:
        results = search_with_tracing(vector_store, collection, query, n_results=1)
        if results['documents'] and results['documents'][0]:
            doc = results['documents'][0][0]
            meta = results['metadatas'][0][0]
            print(f"  Top result: {meta['filename']}")
            print(f"  Preview: {doc[:80]}...")
        else:
            print("  No results in this collection")
    except Exception as e:
        print(f"  Collection not indexed: {e}")

In [ ]:
# ============================================================================
# WEEK 3 CHECKPOINT
# ============================================================================

print("WEEK 3 CHECKPOINT")
print("="*60)

checks = {
    "Vector store initialized": vector_store is not None,
    "Collections created": len(vector_store.collections) > 0,
    "Documents indexed": True,  # Would check collection count
    "Search working": True  # Would verify search results
}

all_passed = True
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")
    if not passed:
        all_passed = False

print(f"\nCollections: {list(vector_store.collections.keys())}")

if all_passed:
    print("\nAll Week 3 checks passed! Ready for Week 4.")

## Week 3 Summary

### What We Accomplished

1. **Vector Store Setup**
   - Initialized ChromaDB with persistence
   - Created collections for different document types
   - Implemented document chunking strategy

2. **Traced Indexing**
   - Every embedding operation logged to Langfuse
   - Document metadata preserved
   - Created unified and specialty collections

3. **Semantic Search**
   - Implemented query embedding with tracing
   - Demonstrated clinical query retrieval
   - Showed similarity scoring

### Key Concepts Learned

- **Embeddings**: Dense vectors capturing semantic meaning
- **Chunking**: Breaking documents for better retrieval
- **RAG**: Combining retrieval with generation

### Next Week Preview

In Week 4, we'll build **specialized clinical agents** for lab interpretation and radiology analysis.

---

---
# WEEK 4: Single Agent Design (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 4: Single Agent Design

---

## Learning Objectives

By the end of Week 4, you will:
- Design specialized clinical agents with Langfuse tracing
- Implement structured output parsing with Pydantic
- Create Lab Interpreter, Radiology Analyzer, and Allergy Assessment agents
- Run multiple agent invocations with full observability

---

## Deep Dive: Clinical AI Safety

Clinical AI systems must be designed with safety as the primary concern.

### The FDA's Clinical Decision Support Framework

The FDA categorizes clinical decision support (CDS) software:

| Category | Description | Regulation |
|----------|-------------|------------|
| **Non-device CDS** | Allows clinician to review basis for recommendation | Exempt |
| **Device CDS** | Provides diagnosis/treatment without explanation | Regulated |

### Safety Design Principles

1. **Transparency**: Always show the evidence behind recommendations
2. **Confidence Scores**: Indicate uncertainty in AI outputs
3. **Fail-Safe Defaults**: When uncertain, recommend human review
4. **Audit Trails**: Log all decisions for review

### Our Agent Design

Each agent in our system:
- Returns structured outputs with confidence scores
- Cites source documents
- Flags critical findings
- Is fully traced to Langfuse

---

In [ ]:
# ============================================================================
# WEEK 4.1: PYDANTIC IMPORTS
# ============================================================================

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

print("Pydantic and LangChain imports loaded")

In [ ]:
# ============================================================================
# WEEK 4.2: LAB INTERPRETATION MODEL
# ============================================================================

class LabInterpretation(BaseModel):
    """
    Structured output for lab result interpretation.

    This model ensures consistent, parseable outputs
    that can be logged, audited, and acted upon.
    """
    test_name: str = Field(description="Name of the lab test")
    result_value: str = Field(description="The actual result value with units")
    status: str = Field(description="Status: NORMAL, LOW, HIGH, CRITICAL_LOW, CRITICAL_HIGH")
    clinical_significance: str = Field(description="What this result means clinically")
    potential_causes: List[str] = Field(description="Possible causes for abnormal results")
    recommended_actions: List[str] = Field(description="Suggested follow-up actions")
    urgency: str = Field(description="Urgency: routine, urgent, stat, emergency")
    confidence: float = Field(ge=0.0, le=1.0, description="Interpretation confidence (0-1)")

print("LabInterpretation model defined")

In [ ]:
# ============================================================================
# WEEK 4.3: RADIOLOGY FINDING MODEL
# ============================================================================

class RadiologyFinding(BaseModel):
    """
    Structured output for radiology interpretation.

    Follows standard radiology reporting structure:
    Indication -> Technique -> Findings -> Impression
    """
    study_type: str = Field(description="Type of imaging study (X-ray, CT, MRI, Ultrasound)")
    body_region: str = Field(description="Body region examined")
    findings: List[str] = Field(description="Key findings from the study")
    impression: str = Field(description="Overall impression/diagnosis")
    differential_diagnoses: List[str] = Field(description="Possible diagnoses to consider")
    recommendations: List[str] = Field(description="Recommended follow-up actions")
    urgency: str = Field(description="Urgency of findings: routine, urgent, stat")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence in interpretation")

print("RadiologyFinding model defined")

In [ ]:
# ============================================================================
# WEEK 4.4: ALLERGY ASSESSMENT MODEL
# ============================================================================

class AllergyAssessment(BaseModel):
    """
    Structured output for allergy safety assessment.

    Critical for patient safety - identifies cross-reactive
    allergens and provides emergency protocols.
    """
    allergen: str = Field(description="The allergen identified")
    category: str = Field(description="Category: Food, Medication, Environmental, Skin")
    severity: str = Field(description="Severity: MILD, MODERATE, SEVERE, CRITICAL")
    reactions: List[str] = Field(description="Known reactions to this allergen")
    cross_reactivities: List[str] = Field(description="Related allergens to avoid")
    safe_alternatives: List[str] = Field(description="Safe alternatives if applicable")
    emergency_protocol: str = Field(description="Emergency response if exposure occurs")
    confidence: float = Field(ge=0.0, le=1.0, description="Assessment confidence")

print("AllergyAssessment model defined")

## Key Concept: Clinical Reference Ranges

Lab interpretation requires understanding reference ranges and clinical context.

### Why Reference Ranges Vary

Reference ranges depend on:
- **Age**: Pediatric vs adult vs geriatric
- **Sex**: Many tests have sex-specific ranges
- **Pregnancy**: Physiologic changes alter many values
- **Lab Method**: Different assays have different ranges

### Critical Values (Panic Values)

Some results require immediate notification:

| Test | Critical Low | Critical High |
|------|-------------|---------------|
| Hemoglobin | <7 g/dL | >20 g/dL |
| Glucose | <40 mg/dL | >500 mg/dL |
| Potassium | <2.5 mEq/L | >6.5 mEq/L |
| Sodium | <120 mEq/L | >160 mEq/L |
| WBC | <2,000 /uL | >30,000 /uL |
| Platelets | <50,000 /uL | >1,000,000 /uL |

---

In [ ]:
# ============================================================================
# WEEK 4.5: TRACED LAB INTERPRETER AGENT
# ============================================================================

class TracedLabInterpreterAgent:
    """
    Lab Interpreter Agent with full Langfuse tracing.

    Interprets laboratory results and provides:
    - Clinical significance
    - Potential causes
    - Recommended actions
    - Urgency assessment

    Every interpretation is logged for audit compliance.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=LabInterpretation)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert clinical laboratory medicine specialist.
Interpret lab results and provide clinical significance, potential causes, and recommendations.

Reference ranges (use these for interpretation):
- Hemoglobin: Male 13.5-17.5 g/dL, Female 12.0-16.0 g/dL
- WBC: 4,500-11,000 /uL
- Platelet: 150,000-400,000 /uL
- Glucose (fasting): 70-100 mg/dL
- Creatinine: 0.7-1.3 mg/dL
- BUN: 7-20 mg/dL
- AST: 10-40 U/L, ALT: 7-56 U/L
- TSH: 0.4-4.0 mIU/L
- Free T4: 0.8-1.8 ng/dL

{format_instructions}"""),
            ("human", "Interpret this lab report:\n\n{lab_report}")
        ])

    def interpret(self, lab_report: str, patient_id: str = "anonymous") -> Dict:
        """
        Interpret lab report with Langfuse tracing.

        Every interpretation is logged with:
        - Input lab report
        - Interpretation result
        - Confidence score
        - Token usage
        """
        handler = get_langfuse_handler(
            trace_name=f"lab-interpretation-{patient_id}",
            tags=["week4", "lab", "agent", "interpretation"]
        )

        llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
        chain = self.prompt | llm | self.parser

        try:
            result = chain.invoke({
                "lab_report": lab_report,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}

print("TracedLabInterpreterAgent class defined")

In [ ]:
# ============================================================================
# WEEK 4.6: TRACED RADIOLOGY AGENT
# ============================================================================

class TracedRadiologyAgent:
    """
    Radiology Analyzer Agent with full Langfuse tracing.

    Analyzes radiology reports and provides:
    - Key findings
    - Differential diagnoses
    - Recommendations
    - Urgency flags for critical findings
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=RadiologyFinding)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert radiologist specializing in diagnostic imaging.
Analyze radiology reports and provide findings, differential diagnoses, and recommendations.
Flag urgent findings requiring immediate attention.

For each study type, consider:
- X-ray: Lung fields, heart size, bones, soft tissues
- CT: Detailed anatomy, contrast patterns, masses
- MRI: Soft tissue detail, brain/spine evaluation
- Ultrasound: Real-time imaging, Doppler flow

{format_instructions}"""),
            ("human", "Analyze this radiology report:\n\n{radiology_report}")
        ])

    def analyze(self, radiology_report: str, patient_id: str = "anonymous") -> Dict:
        """Analyze radiology report with Langfuse tracing."""
        handler = get_langfuse_handler(
            trace_name=f"radiology-analysis-{patient_id}",
            tags=["week4", "radiology", "agent", "imaging"]
        )

        llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
        chain = self.prompt | llm | self.parser

        try:
            result = chain.invoke({
                "radiology_report": radiology_report,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}

print("TracedRadiologyAgent class defined")

In [ ]:
# ============================================================================
# WEEK 4.7: TRACED ALLERGY AGENT
# ============================================================================

class TracedAllergyAgent:
    """
    Allergy Safety Agent with full Langfuse tracing.

    Assesses allergy severity, identifies cross-reactive
    allergens, and provides safety recommendations.

    Patient safety is the top priority.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=AllergyAssessment)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an allergy and immunology specialist.
Assess allergy severity, identify cross-reactive allergens, and provide safety recommendations.
Patient safety is the top priority.

Known cross-reactivities:
- Peanut: tree nuts, legumes, lupin
- Penicillin: amoxicillin, cephalosporins (10% cross-react)
- Latex: banana, avocado, kiwi, chestnut
- Shellfish: shrimp, crab, lobster (crustaceans cross-react)
- Aspirin/NSAIDs: ibuprofen, naproxen

{format_instructions}"""),
            ("human", "Assess this allergy documentation:\n\n{allergy_doc}")
        ])

    def assess(self, allergy_doc: str, patient_id: str = "anonymous") -> Dict:
        """Assess allergy with Langfuse tracing."""
        handler = get_langfuse_handler(
            trace_name=f"allergy-assessment-{patient_id}",
            tags=["week4", "allergy", "agent", "safety"]
        )

        llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
        chain = self.prompt | llm | self.parser

        try:
            result = chain.invoke({
                "allergy_doc": allergy_doc,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}

print("TracedAllergyAgent class defined")

In [ ]:
# ============================================================================
# WEEK 4.8: INITIALIZE AGENTS
# ============================================================================

# Initialize all clinical agents
lab_agent = TracedLabInterpreterAgent()
radiology_agent = TracedRadiologyAgent()
allergy_agent = TracedAllergyAgent()

print("Traced Clinical Agents initialized:")
print("  - TracedLabInterpreterAgent")
print("  - TracedRadiologyAgent")
print("  - TracedAllergyAgent")

In [ ]:
# ============================================================================
# WEEK 4.9: LAB AGENT EXAMPLES - CBC
# ============================================================================

print("LAB INTERPRETER AGENT - CBC Examples")
print("="*60)
print("Each invocation is traced to Langfuse\n")

cbc_examples = [
    """CBC Report - Patient A:
    Hemoglobin: 10.2 g/dL (LOW)
    WBC: 12,500 /uL (HIGH)
    Platelet: 180,000 /uL (NORMAL)""",

    """CBC Report - Patient B:
    Hemoglobin: 15.5 g/dL (NORMAL)
    WBC: 3,200 /uL (LOW)
    Platelet: 95,000 /uL (LOW)"""
]

for i, report in enumerate(cbc_examples, 1):
    print(f"\n[CBC Example {i}]")
    print("-" * 40)

    result = lab_agent.interpret(report, patient_id=f"cbc-{i:03d}")

    if result['success']:
        r = result['result']
        print(f"  Test: {r['test_name']}")
        print(f"  Value: {r['result_value']}")
        print(f"  Status: {r['status']}")
        print(f"  Urgency: {r['urgency']}")
        print(f"  Confidence: {r['confidence']:.2f}")
        print(f"  Significance: {r['clinical_significance'][:100]}...")
    else:
        print(f"  Error: {result['error']}")

In [ ]:
# ============================================================================
# WEEK 4.10: LAB AGENT EXAMPLES - Liver & Kidney
# ============================================================================

print("\nLAB INTERPRETER AGENT - Liver & Kidney Examples")
print("="*60)

organ_examples = [
    """Liver Function Test:
    AST: 85 U/L (HIGH)
    ALT: 92 U/L (HIGH)
    Bilirubin: 1.8 mg/dL (ELEVATED)""",

    """Kidney Function Test:
    Creatinine: 2.1 mg/dL (HIGH)
    BUN: 28 mg/dL (ELEVATED)
    eGFR: 35 mL/min (LOW)""",

    """Thyroid Panel:
    TSH: 8.5 mIU/L (HIGH)
    Free T4: 0.6 ng/dL (LOW)"""
]

for i, report in enumerate(organ_examples, 1):
    print(f"\n[Organ Panel {i}]")
    print("-" * 40)

    result = lab_agent.interpret(report, patient_id=f"organ-{i:03d}")

    if result['success']:
        r = result['result']
        print(f"  Test: {r['test_name']}")
        print(f"  Status: {r['status']} | Urgency: {r['urgency']}")
        print(f"  Potential Causes: {r['potential_causes'][:2]}")
        print(f"  Actions: {r['recommended_actions'][:2]}")

In [ ]:
# ============================================================================
# WEEK 4.11: ALLERGY AGENT EXAMPLES
# ============================================================================

print("\nALLERGY ASSESSMENT AGENT - Multiple Examples")
print("="*60)

allergy_examples = [
    """Patient has documented peanut allergy.
    Previous reaction: Severe anaphylaxis requiring epinephrine.
    Avoid all peanut products and tree nuts.""",

    """Medication allergy to Penicillin.
    Reaction: Skin rash and hives within 2 hours.
    Cross-reactivity with cephalosporins possible.""",

    """Environmental allergy to dust mites.
    Symptoms: Sneezing, runny nose, itchy eyes.
    Triggered by bedding and carpets.""",

    """Latex allergy documented.
    Contact dermatitis on hands after glove use.
    Note: May have cross-reactivity with certain fruits."""
]

for i, doc in enumerate(allergy_examples, 1):
    print(f"\n[Allergy Assessment {i}]")
    print("-" * 40)

    result = allergy_agent.assess(doc, patient_id=f"allergy-{i:03d}")

    if result['success']:
        r = result['result']
        print(f"  Allergen: {r['allergen']}")
        print(f"  Category: {r['category']}")
        print(f"  Severity: {r['severity']}")
        print(f"  Cross-reactivities: {r['cross_reactivities'][:3]}")
        print(f"  Emergency: {r['emergency_protocol'][:80]}...")

In [ ]:
# ============================================================================
# WEEK 4 CHECKPOINT
# ============================================================================

print("WEEK 4 CHECKPOINT")
print("="*60)

checks = {
    "Lab agent initialized": lab_agent is not None,
    "Radiology agent initialized": radiology_agent is not None,
    "Allergy agent initialized": allergy_agent is not None,
    "Pydantic models defined": True,
    "Agents use tracing": True
}

all_passed = True
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")
    if not passed:
        all_passed = False

if all_passed:
    print("\nAll Week 4 checks passed! Ready for Week 5.")

## Week 4 Summary

### What We Accomplished

1. **Structured Output Models**
   - LabInterpretation for lab results
   - RadiologyFinding for imaging
   - AllergyAssessment for safety

2. **Specialized Agents**
   - Lab Interpreter with clinical reference ranges
   - Radiology Analyzer for imaging reports
   - Allergy Agent with cross-reactivity knowledge

3. **Safety Features**
   - Confidence scores on all outputs
   - Urgency classification
   - Full tracing for audit compliance

### Key Concepts Learned

- **Clinical AI Safety**: FDA CDS framework, transparency requirements
- **Reference Ranges**: Understanding lab value interpretation
- **Structured Outputs**: Pydantic for reliable agent responses

### Next Week Preview

In Week 5, we'll build **multi-agent orchestration** with role-based routing.

---

---
# WEEK 5: Multi-Agent Orchestration (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 5: Multi-Agent Orchestration

---

## Learning Objectives

By the end of Week 5, you will:
- Implement orchestration patterns with full tracing
- Build role-based routing for clinical workflows
- Handle inter-agent communication
- Manage clinical access control

---

## Key Concept: Clinical Roles and Access Control

Healthcare systems implement strict role-based access control (RBAC).

### Clinical Roles and Data Access

| Role | Lab Data | Imaging | Allergies | Treatment |
|------|----------|---------|-----------|----------|
| **Patient** | Own results | Limited | Own allergies | View only |
| **Clinician** | Full access | Summary | Full access | Prescribe |
| **Radiologist** | Relevant | Full access | Safety alerts | Report |
| **Coordinator** | Full access | Full access | Full access | Coordinate |

---

In [ ]:
# ============================================================================
# WEEK 5.1: CLINICAL ROLE ENUM
# ============================================================================

class ClinicalRole(Enum):
    """Clinical roles with different access levels."""
    PATIENT = "patient"
    CLINICIAN = "clinician"
    RADIOLOGIST = "radiologist"
    COORDINATOR = "coordinator"

print("ClinicalRole enum defined")

In [ ]:
# ============================================================================
# WEEK 5.2: TRACED CLINICAL ORCHESTRATOR
# ============================================================================

class TracedClinicalOrchestrator:
    """
    Multi-agent orchestrator with full Langfuse tracing.

    Responsibilities:
    - Route queries to appropriate agents
    - Enforce role-based access control
    - Coordinate multi-agent workflows
    - Log all decisions for audit
    """

    def __init__(self):
        self.lab_agent = lab_agent
        self.radiology_agent = radiology_agent
        self.allergy_agent = allergy_agent
        self.vector_store = vector_store

        # Role-based permissions
        self.role_permissions = {
            ClinicalRole.PATIENT: ['allergy'],
            ClinicalRole.CLINICIAN: ['lab', 'allergy'],
            ClinicalRole.RADIOLOGIST: ['radiology'],
            ClinicalRole.COORDINATOR: ['lab', 'radiology', 'allergy']
        }

    def route_query(self, query: str) -> str:
        """Route query to appropriate agent based on content."""
        query_lower = query.lower()

        if any(kw in query_lower for kw in ['lab', 'blood', 'hemoglobin', 'cbc', 'liver', 'kidney', 'glucose']):
            return 'lab'
        elif any(kw in query_lower for kw in ['x-ray', 'ct', 'mri', 'ultrasound', 'imaging', 'scan']):
            return 'radiology'
        elif any(kw in query_lower for kw in ['allergy', 'allergic', 'reaction', 'peanut', 'penicillin']):
            return 'allergy'
        else:
            return 'general'

    def check_permission(self, role: ClinicalRole, agent_type: str) -> bool:
        """Check if role has permission for agent type."""
        if agent_type == 'general':
            return True
        return agent_type in self.role_permissions.get(role, [])

print("TracedClinicalOrchestrator class defined")

In [ ]:
# ============================================================================
# WEEK 5.3: ORCHESTRATOR PROCESS FUNCTION
# ============================================================================

def orchestrator_process(orchestrator, query: str, role: ClinicalRole) -> Dict:
    """Process query through the orchestrator with full tracing."""
    trace = langfuse.trace(
        name=f"orchestration-{role.value}",
        session_id=SESSION_ID,
        input={"query": query, "role": role.value},
        tags=["week5", "orchestration", role.value]
    )

    # Route query
    agent_type = orchestrator.route_query(query)

    # Check permissions
    has_permission = orchestrator.check_permission(role, agent_type)

    if not has_permission:
        trace.update(output={"error": "access_denied"})
        return {
            'error': f'Role {role.value} lacks permission for {agent_type}',
            'routing': agent_type,
            'allowed': orchestrator.role_permissions.get(role, [])
        }

    # Retrieve context (RAG)
    context_text = ""
    sources = []

    try:
        results = search_with_tracing(orchestrator.vector_store, "healthcare_all", query, n_results=3)
        if results['documents'] and results['documents'][0]:
            context_text = "\n\n".join(results['documents'][0][:2])
            sources = [m['filename'] for m in results['metadatas'][0]]
    except:
        pass

    trace.update(output={"routing": agent_type, "role": role.value})

    return {
        'query': query,
        'role': role.value,
        'routing': agent_type,
        'context': context_text[:500] if context_text else "No context found",
        'sources': sources
    }

print("orchestrator_process() function defined")

In [ ]:
# ============================================================================
# WEEK 5.4: INITIALIZE ORCHESTRATOR
# ============================================================================

orchestrator = TracedClinicalOrchestrator()
print("Traced Clinical Orchestrator initialized")
print(f"\nRole permissions:")
for role, perms in orchestrator.role_permissions.items():
    print(f"  {role.value}: {perms}")

In [ ]:
# ============================================================================
# WEEK 5.5: ORCHESTRATION EXAMPLES
# ============================================================================

print("ORCHESTRATION EXAMPLES - Role-Based Access")
print("="*60)

orchestration_tests = [
    ("What are my peanut allergy precautions?", ClinicalRole.PATIENT),
    ("Interpret my blood test results", ClinicalRole.PATIENT),
    ("What does elevated hemoglobin mean?", ClinicalRole.CLINICIAN),
    ("Chest X-ray findings for pneumonia", ClinicalRole.RADIOLOGIST),
    ("Interpret the liver function test", ClinicalRole.RADIOLOGIST),
    ("Full patient workup summary", ClinicalRole.COORDINATOR)
]

for i, (query, role) in enumerate(orchestration_tests, 1):
    print(f"\n[Test {i}] Role: {role.value}")
    print(f"Query: '{query}'")

    result = orchestrator_process(orchestrator, query, role)

    if 'error' in result:
        print(f"  ACCESS DENIED: {result['error']}")
    else:
        print(f"  ACCESS GRANTED - Routing: {result['routing']}")

## Week 5 Summary

### What We Accomplished

1. **Role-Based Access Control** - Defined clinical roles with permissions
2. **Query Routing** - Keyword-based routing to appropriate agents
3. **RAG Integration** - Retrieved context for each query

---

---
# WEEK 6: Medical Knowledge Graphs (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 6: Medical Knowledge Graphs

---

## Learning Objectives

- Build a medical knowledge graph using NetworkX
- Model relationships between symptoms, diagnoses, and treatments
- Implement allergy cross-reactivity reasoning

---

In [ ]:
# ============================================================================
# WEEK 6.1: KNOWLEDGE GRAPH IMPORTS
# ============================================================================

import networkx as nx
from pyvis.network import Network

print("NetworkX and PyVis imported")

In [ ]:
# ============================================================================
# WEEK 6.2: MEDICAL KNOWLEDGE GRAPH CLASS
# ============================================================================

class MedicalKnowledgeGraph:
    """Knowledge graph for medical reasoning."""

    def __init__(self):
        self.graph = nx.DiGraph()
        self._build_medical_ontology()

    def _build_medical_ontology(self):
        """Build base medical knowledge graph."""

        # Lab Tests -> Diagnoses
        lab_tests = {
            'CBC': ['anemia', 'infection', 'leukemia'],
            'LFT': ['hepatitis', 'cirrhosis', 'fatty_liver'],
            'KFT': ['kidney_disease', 'dehydration'],
            'Thyroid_Panel': ['hypothyroidism', 'hyperthyroidism']
        }

        for test, diagnoses in lab_tests.items():
            self.graph.add_node(test, type='TEST')
            for diagnosis in diagnoses:
                self.graph.add_node(diagnosis, type='DIAGNOSIS')
                self.graph.add_edge(test, diagnosis, relation='CONFIRMS')

        # Symptoms -> Diagnoses
        symptom_diagnosis = {
            'fatigue': ['anemia', 'hypothyroidism', 'infection'],
            'fever': ['infection', 'inflammation'],
            'shortness_of_breath': ['anemia', 'pneumonia', 'heart_failure'],
            'cough': ['pneumonia', 'bronchitis', 'asthma']
        }

        for symptom, diagnoses in symptom_diagnosis.items():
            self.graph.add_node(symptom, type='SYMPTOM')
            for diagnosis in diagnoses:
                if not self.graph.has_node(diagnosis):
                    self.graph.add_node(diagnosis, type='DIAGNOSIS')
                self.graph.add_edge(symptom, diagnosis, relation='INDICATES')

        # Allergen cross-reactivity
        allergen_cross = {
            'peanut': ['tree_nuts', 'legumes', 'lupin'],
            'penicillin': ['amoxicillin', 'cephalosporins'],
            'latex': ['banana', 'avocado', 'kiwi'],
            'shellfish': ['shrimp', 'crab', 'lobster'],
            'aspirin': ['nsaids', 'ibuprofen']
        }

        for allergen, cross_reactive in allergen_cross.items():
            self.graph.add_node(allergen, type='ALLERGEN')
            for related in cross_reactive:
                self.graph.add_node(related, type='ALLERGEN')
                self.graph.add_edge(allergen, related, relation='CROSS_REACTIVE')

        # Treatments
        diagnosis_treatment = {
            'anemia': ['iron_supplements', 'b12_injection'],
            'infection': ['antibiotics', 'antivirals'],
            'hypothyroidism': ['levothyroxine']
        }

        for diagnosis, treatments in diagnosis_treatment.items():
            for treatment in treatments:
                self.graph.add_node(treatment, type='TREATMENT')
                self.graph.add_edge(treatment, diagnosis, relation='TREATS')

print("MedicalKnowledgeGraph class defined")

In [ ]:
# ============================================================================
# WEEK 6.3: KNOWLEDGE GRAPH QUERY FUNCTIONS
# ============================================================================

def find_diagnoses_for_symptoms(kg, symptoms):
    """Find possible diagnoses based on symptoms."""
    diagnosis_scores = {}
    for symptom in symptoms:
        if kg.graph.has_node(symptom):
            for _, target, data in kg.graph.out_edges(symptom, data=True):
                if data.get('relation') == 'INDICATES':
                    diagnosis_scores[target] = diagnosis_scores.get(target, 0) + 1
    return {'symptoms': symptoms, 'diagnoses': sorted(diagnosis_scores.items(), key=lambda x: -x[1])}

def get_cross_reactive_allergens(kg, allergen):
    """Get cross-reactive allergens for safety warnings."""
    cross_reactive = []
    if kg.graph.has_node(allergen):
        for _, target, data in kg.graph.out_edges(allergen, data=True):
            if data.get('relation') == 'CROSS_REACTIVE':
                cross_reactive.append(target)
    return cross_reactive

print("Knowledge graph query functions defined")

In [ ]:
# ============================================================================
# WEEK 6.4: INITIALIZE AND EXPLORE KNOWLEDGE GRAPH
# ============================================================================

medical_kg = MedicalKnowledgeGraph()

print("MEDICAL KNOWLEDGE GRAPH")
print("="*60)
print(f"Total nodes: {medical_kg.graph.number_of_nodes()}")
print(f"Total edges: {medical_kg.graph.number_of_edges()}")

# Count by type
node_types = {}
for _, data in medical_kg.graph.nodes(data=True):
    t = data.get('type', 'OTHER')
    node_types[t] = node_types.get(t, 0) + 1

print(f"\nNodes by type:")
for t, count in sorted(node_types.items()):
    print(f"  {t}: {count}")

In [ ]:
# ============================================================================
# WEEK 6.5: KNOWLEDGE GRAPH REASONING EXAMPLES
# ============================================================================

print("\nKNOWLEDGE GRAPH REASONING EXAMPLES")
print("="*60)

# Symptom-to-Diagnosis
print("\n1. Symptom-to-Diagnosis Reasoning:")
for symptoms in [['fatigue', 'shortness_of_breath'], ['fever', 'cough']]:
    result = find_diagnoses_for_symptoms(medical_kg, symptoms)
    print(f"   Symptoms: {symptoms}")
    print(f"   Diagnoses: {result['diagnoses'][:3]}")

# Cross-Reactivity
print("\n2. Allergy Cross-Reactivity:")
for allergen in ['peanut', 'penicillin', 'latex']:
    cross = get_cross_reactive_allergens(medical_kg, allergen)
    print(f"   {allergen}: avoid {cross}")

In [ ]:
# ============================================================================
# WEEK 6.6: KNOWLEDGE GRAPH VISUALIZATION
# ============================================================================

print("\nKNOWLEDGE GRAPH VISUALIZATION")
print("="*60)

color_map = {
    'SYMPTOM': '#ff6b6b',
    'DIAGNOSIS': '#4ecdc4',
    'TEST': '#45b7d1',
    'TREATMENT': '#96ceb4',
    'ALLERGEN': '#ffeaa7'
}

node_colors = [color_map.get(medical_kg.graph.nodes[n].get('type', 'OTHER'), '#95a5a6')
               for n in medical_kg.graph.nodes()]

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(medical_kg.graph, k=2, iterations=50, seed=42)

nx.draw_networkx_nodes(medical_kg.graph, pos, node_color=node_colors, node_size=500, alpha=0.9)
nx.draw_networkx_edges(medical_kg.graph, pos, edge_color='gray', arrows=True, arrowsize=10, alpha=0.5)
nx.draw_networkx_labels(medical_kg.graph, pos, font_size=7)

legend_elements = [plt.scatter([], [], c=color, s=100, label=node_type)
                   for node_type, color in color_map.items()]
plt.legend(handles=legend_elements, loc='upper left', title='Node Types')

plt.title('Medical Knowledge Graph', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

## Week 6 Summary

### What We Built

1. **Medical Knowledge Graph** with symptoms, diagnoses, tests, treatments, allergens
2. **Clinical Reasoning** - symptom-to-diagnosis inference
3. **Safety Features** - allergen cross-reactivity lookup

---

---
# WEEK 7: Advanced System Integration (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEKS 7-8: Production Integration and Deployment

---

## Learning Objectives

- Integrate all components into a production system
- Create an interactive Gradio interface
- Implement complete end-to-end tracing

---

In [ ]:
# ============================================================================
# WEEK 7-8: HEALTHCARE CLINICAL SYSTEM
# ============================================================================

class HealthcareClinicalSystem:
    """Production-grade Healthcare Clinical Intelligence System."""

    def __init__(self):
        self.vector_store = vector_store
        self.orchestrator = orchestrator
        self.knowledge_graph = medical_kg
        self.lab_agent = lab_agent
        self.radiology_agent = radiology_agent
        self.allergy_agent = allergy_agent

    def process_query(self, query: str, role: str) -> Dict:
        """Process clinical query through full pipeline."""
        try:
            role_enum = ClinicalRole(role.lower())
        except ValueError:
            role_enum = ClinicalRole.PATIENT
        return orchestrator_process(self.orchestrator, query, role_enum)

    def get_cross_reactivity(self, allergen: str) -> List[str]:
        """Get allergen cross-reactivity warnings."""
        return get_cross_reactive_allergens(self.knowledge_graph, allergen)

    def get_status(self) -> Dict:
        """Get system status."""
        return {
            'session_id': SESSION_ID,
            'kg_nodes': self.knowledge_graph.graph.number_of_nodes(),
            'kg_edges': self.knowledge_graph.graph.number_of_edges()
        }

healthcare_system = HealthcareClinicalSystem()

print("HEALTHCARE CLINICAL INTELLIGENCE SYSTEM")
print("="*60)
status = healthcare_system.get_status()
print(f"Session: {status['session_id']}")
print(f"Knowledge Graph: {status['kg_nodes']} nodes, {status['kg_edges']} edges")

In [ ]:
# ============================================================================
# WEEK 8: GRADIO INTERFACE
# ============================================================================

import gradio as gr

def clinical_query_interface(query: str, role: str) -> str:
    """Process clinical query and return formatted response."""
    result = healthcare_system.process_query(query, role)

    if 'error' in result:
        return f"## Access Denied\n\n**Error:** {result['error']}\n\n**Allowed for {role}:** {result.get('allowed', [])}"

    sources_text = '\n'.join([f'- {s}' for s in result['sources']]) if result['sources'] else 'No sources'

    response = f"""## Query Analysis

**Query:** {result['query']}

**Role:** {result['role']}

**Routing:** {result['routing']}

---

### Retrieved Context

{result['context']}

---

### Sources

{sources_text}
"""
    return response


def allergy_check_interface(allergen: str) -> str:
    """Check allergen cross-reactivity."""
    cross_reactive = healthcare_system.get_cross_reactivity(allergen.lower())

    if cross_reactive:
        items = '\n'.join([f'- {a}' for a in cross_reactive])
        return f"## Allergy Cross-Reactivity Alert\n\n**Primary Allergen:** {allergen}\n\n**Also Avoid:**\n{items}"
    else:
        return f"No cross-reactivity data found for: {allergen}"

print("Gradio interface functions defined")

In [ ]:
# ============================================================================
# WEEK 8: LAUNCH GRADIO INTERFACE
# ============================================================================

with gr.Blocks(title="Healthcare Clinical Intelligence") as demo:
    gr.Markdown("# Healthcare Clinical Intelligence System")
    gr.Markdown("Multi-agent AI for clinical decision support with full observability")

    with gr.Tab("Clinical Query"):
        with gr.Row():
            query_input = gr.Textbox(label="Clinical Query", placeholder="Enter your question...", lines=3)
            role_input = gr.Dropdown(choices=["patient", "clinician", "radiologist", "coordinator"], value="coordinator", label="Role")
        query_output = gr.Markdown(label="Response")
        query_btn = gr.Button("Submit Query")
        query_btn.click(clinical_query_interface, [query_input, role_input], query_output)

        gr.Examples(
            examples=[
                ["What do elevated hemoglobin levels indicate?", "clinician"],
                ["Cross-reactive allergens for peanut allergy?", "patient"],
                ["Chest X-ray findings for pneumonia", "radiologist"]
            ],
            inputs=[query_input, role_input]
        )

    with gr.Tab("Allergy Safety Check"):
        allergen_input = gr.Textbox(label="Allergen", placeholder="e.g., peanut, penicillin")
        allergen_output = gr.Markdown(label="Cross-Reactivity Alert")
        allergen_btn = gr.Button("Check Cross-Reactivity")
        allergen_btn.click(allergy_check_interface, allergen_input, allergen_output)

print("Gradio interface created")

if IN_COLAB:
    demo.launch(share=True)
else:
    print("Run demo.launch() to start the interface")

---
# WEEK 8: Production Deployment & UI (YOUR WORK)
---

## Instructions

Complete the TODO cells below. Each cell has hints and instructions.
Refer to the learning objectives above and the master notebook for guidance.

---## Production Deployment: FastAPI ApplicationNow that we have a working Gradio prototype, let's create a **production-ready deployment** using:- **FastAPI** — Industry-standard async API framework- **Animated Web UI** — Beautiful interface showing the RAG pipeline flow in real-time- **Docker** — Containerized deployment ready for AWS/GCPThe web interface features animated visualizations of:- Document processing and chunking- Vector embedding and semantic search- Multi-agent orchestration and routing- Response generation pipeline---

In [ ]:
# ============================================================================
# TODO 8.1: PRODUCTION DEPLOYMENT: FastAPI Application
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement PRODUCTION DEPLOYMENT: FastAPI Application
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.2: ---------------------------------------------------------------------------
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
# HINT: def _lf_trace(name: str, **kwargs):
# HINT: def _lf_generation(trace, name: str, model: str, input_data):
# HINT: def _effective_price(p: dict) -> float:
# HINT: def _product_text(p: dict) -> str:
# HINT: def _init_vector_store():
# HINT: def _batch_embed(texts: List[str]) -> List[List[float]]:
# HINT: def _embed(text: str) -> List[float]:
# HINT: def _search(query: str, n: int = 5, where: dict = None) -> List[dict]:
# HINT: def _traced_completion(messages: List[dict], trace_name: str, model: str = "gpt-4o-mini") -> str:
# HINT: def _run_price_agent(category: str, context: str) -> dict:
# HINT: def _run_catalog_agent(category: str, context: str) -> dict:
# HINT: def _run_marketing_agent(query: str, context: str) -> dict:
# HINT: def run_pipeline(query: str) -> dict:
# HINT: class QueryRequest(BaseModel):
# HINT: class QueryResponse(BaseModel):
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement ---------------------------------------------------------------------------
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.3: Week 8 Task 3
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 3
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.4: Week 8 Task 4
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 4
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.5: Week 8 Task 5
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 5
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.6: Week 8 Task 6
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 6
raise NotImplementedError('Complete this section')

---

## Production Deployment: AWS EC2 Guide

After running the FastAPI cells above, you have a `deployment/` folder with everything needed. Here's how to deploy it to AWS EC2.

### Step 1: Launch an EC2 Instance

1. Go to **AWS Console → EC2 → Launch Instance**
2. **AMI**: Ubuntu Server 22.04 LTS (Free Tier eligible)
3. **Instance type**: `t2.micro` (Free Tier) for testing, `t3.small` or `t3.medium` for production
4. **Key pair**: Create or select an existing `.pem` key pair — download it
5. **Network settings**:
   - Allow SSH (port 22) from your IP
   - Add a **Custom TCP rule** for **port 8000** from `0.0.0.0/0` (or your IP for security)
6. **Storage**: 20 GB gp3 (default is fine)
7. Click **Launch Instance**

### Step 2: Transfer Files to EC2

**Option A: From your local machine (after downloading deployment/ from Colab)**

```bash
# Make your key file secure
chmod 400 your-key.pem

# Copy the deployment folder to EC2
scp -i your-key.pem -r deployment/ ubuntu@<EC2-PUBLIC-IP>:~/app/
```

**Option B: From Google Colab**

```python
# In Colab, zip the deployment folder
!zip -r deployment.zip deployment/

# Download it to your local machine
from google.colab import files
files.download('deployment.zip')

# Then scp from local to EC2 as shown in Option A
```

### Step 3: SSH into EC2 and Set Up

```bash
# Connect to your instance
ssh -i your-key.pem ubuntu@<EC2-PUBLIC-IP>

# Update system and install Python
sudo apt update && sudo apt upgrade -y
sudo apt install -y python3-pip python3-venv

# Set up the app
cd ~/app
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```

### Step 4: Configure Environment Variables

```bash
# Create .env file with your API keys
cat > .env << 'EOF'
OPENAI_API_KEY=sk-your-openai-key
LANGFUSE_SECRET_KEY=sk-lf-your-secret
LANGFUSE_PUBLIC_KEY=pk-lf-your-public
LANGFUSE_HOST=https://cloud.langfuse.com
EOF
```

### Step 5: Launch the Application

```bash
# Test run (foreground)
uvicorn app:app --host 0.0.0.0 --port 8000

# Production run (background, survives SSH disconnect)
nohup uvicorn app:app --host 0.0.0.0 --port 8000 > app.log 2>&1 &

# Or use systemd for auto-restart:
sudo tee /etc/systemd/system/fastapi-app.service << 'EOF'
[Unit]
Description=FastAPI Competitive Intelligence App
After=network.target

[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/app
Environment="PATH=/home/ubuntu/app/venv/bin"
EnvironmentFile=/home/ubuntu/app/.env
ExecStart=/home/ubuntu/app/venv/bin/uvicorn app:app --host 0.0.0.0 --port 8000
Restart=always

[Install]
WantedBy=multi-user.target
EOF

sudo systemctl enable fastapi-app
sudo systemctl start fastapi-app
sudo systemctl status fastapi-app
```

### Step 6: Access Your Application

Open in your browser:
```
http://<EC2-PUBLIC-IP>:8000
```

API endpoints:
- **Health check**: `http://<EC2-PUBLIC-IP>:8000/api/health`
- **Query API**: `POST http://<EC2-PUBLIC-IP>:8000/api/query`
- **Swagger docs**: `http://<EC2-PUBLIC-IP>:8000/docs`

### Step 7: Verify Deployment

```bash
# From your local machine or any terminal
curl http://<EC2-PUBLIC-IP>:8000/api/health

# Test a query
curl -X POST http://<EC2-PUBLIC-IP>:8000/api/query \
  -H "Content-Type: application/json" \
  -d '{"query": "Analyze competitive pricing"}'
```

### Docker Alternative (Optional)

If you prefer Docker:

```bash
# On EC2
sudo apt install -y docker.io
sudo usermod -aG docker ubuntu
# Log out and back in, then:
cd ~/app
docker build -t intelligence-app .
docker run -d -p 8000:8000 --env-file .env --name app intelligence-app
```

### Security Recommendations for Production

- Use HTTPS with a reverse proxy (nginx + Let's Encrypt)
- Restrict port 8000 to your IP or use a load balancer
- Store API keys in AWS Secrets Manager instead of .env
- Set up CloudWatch monitoring for the EC2 instance
- Use an Elastic IP so the address doesn't change on reboot


In [ ]:
# ============================================================================
# TODO 8.7: DEPLOYMENT INSTRUCTIONS
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement DEPLOYMENT INSTRUCTIONS
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.8: FINAL CLEANUP
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement FINAL CLEANUP
raise NotImplementedError('Complete this section')

## Course Summary

### What We Built Over 8 Weeks

1. **Week 1-2**: Environment setup, document processing, entity extraction
2. **Week 3**: ChromaDB vector store with traced semantic search
3. **Week 4**: Specialized clinical agents (Lab, Radiology, Allergy)
4. **Week 5**: Multi-agent orchestration with role-based access control
5. **Week 6**: Medical knowledge graph for clinical reasoning
6. **Weeks 7-8**: Full system integration with Gradio UI

### Key Takeaways

- **Safety First**: Clinical AI requires transparency, traceability, and fail-safes
- **Observability**: Every operation traced for audit and debugging
- **Domain Knowledge**: Understanding clinical concepts is as important as technical skills
- **Access Control**: RBAC is essential for HIPAA compliance

---

**Congratulations on completing the Healthcare Clinical Intelligence Capstone!**